# 네트워크 서비스 카테고리

## ENI(Elastic Network Interface)

- 가상 네트워크 인터페이스 카드
  - 모든 VM 인스턴스는 네트워킹을 위한 가상의 네트워크 인터페이스 카드(vnic: virtual network interface card)를 하나 이상 가짐
- ENI
  - AWS에서는 이 가상 네트워크 인터페이스를 ENI라고 함
  - 하나의 ENI는 하나의 서브넷에 붙음
- Primary ENI
  - 인스턴스 생성시에 하나의 ENI를 자동 생성하는데 이름 Primary ENI라고 함
- Secondary ENI
  - 하나의 인스턴스가 여러 네트워크(서브넷)에 동시 접속하려면 추가적인 ENI 필요
  - 추가 ENI는 사용자가 수동으로 생성하여 인스턴스에 붙여야 함

## VPC

- 독립적인 가상 네트워크
- 하나의 리전에 5개까지의 VPC 생성 가능
- IP 할당 가능 대역
  - RFC1918 사설 IPv4 대역 사용 권장
    - 10.0.0.0 (사설 클래스 A)
    - 172.16.0.0 (사설 클래스 B)
    - 192.168.0.0 (사설 클래스 C)
  - IP 범위는 `/16`~`/28` 만 지정 가능
    - 클래스 A 전체 (10.0.0.0/8)또는 클래스 B 전체(172.16.0.0/12)를 사용하는 것은 불가
    - 10.0.0.0/16, 10.1.0.0/16, 10.2.0.0/16, ... 이런 식으로 /16 구간을 선택해서 사용 가능


## 서브넷

- VPC의 IP 영역을 다시 서브넷으로 나눌 수 있다.
  - 서브넷은 반드시 하나 이상 존재해야 함
  - 나눌 필요가 없을 경우에는 VPC의 IP 영역과 동일한 영역의 서브넷을 만들어야 함
- 서브넷에서 인스턴스에 사용 불가능한 아이피
  - 시작 아이피: 네트워크
  - 시작 아이피 + `1`: 가상 라우터
  - 시작 아이피 + `2`: 가상 DNS
  - 시작 아이피 + `3`: AWS가 예약해 놓은 아이피
  - 마지막 아이피: 브로드캐스트
- 서브넷 단위로 하나의 라우트 테이블을 연동하여 서브넷에 연결된 모든 인스턴스의 라우팅을 제어 가능


## 라우트 테이블

- AWS의 서브넷 하나는 반드시 하나의 라우트 테이블과 연동되어야 함
  - 반대로 하나의 라우트 테이블은 여러개의 서브넷에 연동될 수 있음
  - 서브넷에 포함된 인스턴스는 해당 서브넷에 연동된 라우트 테이블 규칙을 따라야 함
- 따라서 모든 인스턴스는 사실상 2개의 라우트 테이블 규칙을 따름
  - 하나는 인스턴스 내에 있는 OS 레벨의 라우트 테이블
    - ENI가 여러개인 경우 목적지에 따라 어느 ENI를 사용할지 결정
  - 다른 하나는 인스턴스가 속하는 서브넷의 라우트 테이블
    - 하나의 서브넷에 연결된 모든 ENI에 공통 적용
- 라우트 테이블은 "목적지"와 "타겟"을 가짐
  - 목적지는 데이터가 최종적으로 도착할 IP 주소
  - 타겟은 해앙 데이터를 물리적으로 건네줄 기기
- AWS 라우트 테이블 타겟
  - OS 레벨의 라우트 테이블과 달리 타겟이 아이피나 디바이스가 아닌 리소스 아이디를 사용
  - 하나의 VPC는 모두 연결되어 있으므로 `local` 타겟 사용
  - 인터넷 등 외부로 나가려면 IGW 또는 NAT의 리소스 아이디를 타겟으로 지정
- 라우트 테이블을 생성하면 항상 자동으로 VPC local 레코드를 추가함
  - 하나의 VPC 내에서는 서브넷이 다르더라도 항상 서로 통신 가능
- 인터넷을 연결하려면 `0.0.0.0/0` 목적지를 IGW로 연결하는 레코드를 명시적으로 추가해야 함
- 메인 라우트 테이블 (main route table)
  - VPC를 만드는 순간에 AWS가 자동으로 만드는 라우트 테이블
  - VPC local을 연결하는 레코드만 존재
  - 콘솔에는 어떤 서브넷과도 연결되어 있지 않다고 표시됨
  - 그러나 사용자가 명시적으로 라우트 테이블에 연결하지 않은 서브넷이 있을 경우, 메인 라우트 테이블이 폴백(fall-back)으로 사용됨